# Chapter 4: Optuna — Automated, Intelligent Hyperparameter Tuning

### Why This Matters

Volume 05's `GridSearchCV` tries *every* combination in a predefined grid — thorough, but wasteful, since it spends equal effort on obviously-bad combinations as on promising ones. **Optuna** treats hyperparameter tuning as a genuine optimization problem: it uses the results of earlier trials to intelligently choose which combination to try next, focusing search effort on the most promising regions of the hyperparameter space, and can *prune* (abandon early) trials that are clearly underperforming partway through — extracting far better results from the same compute budget.

### 4.1 Defining an Objective Function

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    model = LGBMClassifier(**params, random_state=42, verbose=-1, n_jobs=-1)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=-1).mean()
    return score       # Optuna will try to MAXIMIZE (or minimize) whatever this function returns

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=20, show_progress_bar=False)

print("Best ROC-AUC:", round(study.best_value, 4))
print("Best params:", study.best_params)

An **objective function** is the one piece of code you have to write yourself: given a `trial` (Optuna's handle for "one attempt, with one specific set of hyperparameter values"), build a model with those values, evaluate it (here, with 3-fold cross-validation — always evaluate honestly, never on the same data used to pick the hyperparameters), and return the score. `trial.suggest_int`, `trial.suggest_float`, and `trial.suggest_categorical` define the *search space* for each hyperparameter — the range Optuna is allowed to explore. `log=True` on `learning_rate` tells Optuna to search that range on a logarithmic scale rather than linear, which matters because the practical difference between a learning rate of 0.01 and 0.02 is much larger than the difference between 0.11 and 0.12, even though both pairs are separated by 0.01 in absolute terms.

`optuna.create_study(direction="maximize", ...)` sets up the overall search; `.optimize(objective, n_trials=20)` actually runs it, calling your objective function 20 times with 20 different (intelligently chosen) hyperparameter combinations. `MedianPruner` is what enables early abandonment of clearly weak trials partway through — genuinely valuable when each individual trial is expensive (e.g., training a large model for many epochs), less critical for the fast tabular models in this volume.

⚠️ **Never tune hyperparameters against your final test set.** Both `GridSearchCV` and this Optuna example use cross-validation *within the training data* to select hyperparameters — the test set (`X_test`/`y_test`) should remain completely untouched until the very final evaluation of your chosen, already-tuned model. Using the test set to guide any decision during tuning — even innocuously, "let me just check how this configuration does on the test set" — silently leaks information and produces an overly optimistic final performance estimate that won't hold up once genuinely new data arrives.

### Cheat Sheet — Optuna

| Task | Code |
|---|---|
| Define search space | `trial.suggest_int/float/categorical(name, low, high)` |
| Log-scale search | `trial.suggest_float(name, low, high, log=True)` |
| Create a study | `optuna.create_study(direction="maximize")` |
| Run the search | `study.optimize(objective, n_trials=N)` |
| Best result | `study.best_value`, `study.best_params` |
| Early-stop weak trials | `pruner=optuna.pruners.MedianPruner()` |

---

## Volume 06 — What You Should Be Able to Do Now

- Explain, in your own words, why gradient boosting builds trees sequentially rather than independently
- Train and compare XGBoost, LightGBM, and CatBoost on the same tabular dataset using ROC-AUC
- Explain why CatBoost can skip manual one-hot encoding, and when that advantage matters most
- Write an Optuna objective function and run a tuning study, correctly using cross-validation rather than the test set
- Explain the `learning_rate` / `n_estimators` tradeoff and why both must be considered together

**Next: `07_AutoML_Toolkit.md`**